# Extracting Camera Intrinsic

## The Intrinsic Matrix ($K$)

The data we are extracting forms a $3 \times 3$ matrix called the Camera Matrix ($K$), which OpenCV requires for the next step (the PnP math). It looks like this:
$$K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}$$
$f_x, f_y$ (Focal Lengths): Represented in pixels. If the lens is perfectly symmetrical, $f_x$ and $f_y$ will be almost identical.
$c_x, c_y$ (Principal Point): The exact pixel coordinate where the optical axis intersects the image sensor. For your 1280x720 resolution (noted in your SAM3 notebook), this should be roughly at pixel (640, 360).

In [1]:
from pathlib import Path

import cv2
import numpy as np
import pyrealsense2 as rs

def probe_v4l_node(device_path):
    capture = cv2.VideoCapture(str(device_path), cv2.CAP_V4L2)
    if not capture.isOpened():
        capture.release()
        return {"opens": False}

    capture.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    capture.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    capture.set(cv2.CAP_PROP_FPS, 15)
    capture.set(cv2.CAP_PROP_CONVERT_RGB, 0)

    raw_frame = None
    for _ in range(8):
        ok, frame = capture.read()
        if ok and frame is not None:
            raw_frame = frame.copy()

    fourcc = int(capture.get(cv2.CAP_PROP_FOURCC))
    fourcc_text = "".join(chr((fourcc >> (8 * byte_index)) & 0xFF) for byte_index in range(4))
    capture.release()

    if raw_frame is None:
        return {"opens": True, "reads": False, "fourcc": fourcc_text}

    probe = {
        "opens": True,
        "reads": True,
        "fourcc": fourcc_text,
        "raw_shape": tuple(raw_frame.shape),
    }

    if raw_frame.ndim == 3 and raw_frame.shape[2] == 2:
        decode_summary = {}
        for decode_name, decode_code in (
            ("UYVY", cv2.COLOR_YUV2BGR_UYVY),
            ("YUY2", cv2.COLOR_YUV2BGR_YUY2),
        ):
            decoded = cv2.cvtColor(raw_frame, decode_code)
            channel_spread = float(
                np.mean(np.abs(decoded[:, :, 0].astype(np.int16) - decoded[:, :, 1].astype(np.int16)))
                + np.mean(np.abs(decoded[:, :, 1].astype(np.int16) - decoded[:, :, 2].astype(np.int16)))
            )
            decode_summary[decode_name] = {
                "channel_spread": round(channel_spread, 1),
                "mean_bgr": decoded.reshape(-1, 3).mean(axis=0).round(1).tolist(),
            }

        best_decode_name = max(
            decode_summary, key=lambda name: decode_summary[name]["channel_spread"]
        )
        probe["best_decode"] = best_decode_name
        probe["best_channel_spread"] = decode_summary[best_decode_name]["channel_spread"]
        probe["best_mean_bgr"] = decode_summary[best_decode_name]["mean_bgr"]
        probe["looks_color"] = probe["best_channel_spread"] >= 20.0
    elif raw_frame.ndim == 3 and raw_frame.shape[2] == 3:
        channel_spread = float(
            np.mean(np.abs(raw_frame[:, :, 0].astype(np.int16) - raw_frame[:, :, 1].astype(np.int16)))
            + np.mean(np.abs(raw_frame[:, :, 1].astype(np.int16) - raw_frame[:, :, 2].astype(np.int16)))
        )
        probe["best_decode"] = "native-bgr"
        probe["best_channel_spread"] = round(channel_spread, 1)
        probe["best_mean_bgr"] = raw_frame.reshape(-1, 3).mean(axis=0).round(1).tolist()
        probe["looks_color"] = True
    else:
        probe["best_decode"] = None
        probe["best_channel_spread"] = None
        probe["best_mean_bgr"] = None
        probe["looks_color"] = False

    return probe

def collect_realsense_v4l_nodes():
    nodes = []
    for name_file in sorted(Path('/sys/class/video4linux').glob('video*/name')):
        try:
            card_name = name_file.read_text().strip()
        except OSError:
            continue
        if 'Intel(R) RealSense' not in card_name:
            continue

        device_path = Path('/dev') / name_file.parent.name
        nodes.append(
            {
                "node": name_file.parent.name,
                "path": str(device_path),
                "present": device_path.exists(),
                "card_name": card_name,
            }
        )
    return nodes

ctx = rs.context()
devices = list(ctx.query_devices())
device_summaries = []
for device in devices:
    device_summaries.append(
        {
            "name": device.get_info(rs.camera_info.name) if device.supports(rs.camera_info.name) else "unknown",
            "serial": device.get_info(rs.camera_info.serial_number) if device.supports(rs.camera_info.serial_number) else "unknown",
            "sensors": [
                sensor.get_info(rs.camera_info.name)
                for sensor in device.query_sensors()
                if sensor.supports(rs.camera_info.name)
            ],
        }
    )

realsense_v4l_nodes = collect_realsense_v4l_nodes()
v4l_probe_results = []
for node_info in realsense_v4l_nodes:
    if not node_info["present"]:
        continue
    probe_result = probe_v4l_node(Path(node_info["path"]))
    v4l_probe_results.append({"path": node_info["path"], **probe_result})

print(f"RealSense SDK devices visible: {len(device_summaries)}")
if device_summaries:
    for index, summary in enumerate(device_summaries, start=1):
        print(
            f"  Device {index}: {summary['name']} | serial={summary['serial']} | sensors={summary['sensors']}"
        )
else:
    print("  No devices were returned by pyrealsense2 in this container.")

print(
    "RealSense V4L nodes: "
    + str([
        (node_info["node"], node_info["present"], node_info["card_name"])
        for node_info in realsense_v4l_nodes
    ])
)
print("Container V4L probe:")
if not v4l_probe_results:
    print("  No RealSense /dev/video* nodes are present inside the container.")
else:
    for probe_result in v4l_probe_results:
        print(
            "  "
            + f"{probe_result['path']}: opens={probe_result.get('opens', False)}, "
            + f"reads={probe_result.get('reads', False)}, "
            + f"fourcc={probe_result.get('fourcc')}, "
            + f"best_decode={probe_result.get('best_decode')}, "
            + f"channel_spread={probe_result.get('best_channel_spread')}, "
            + f"looks_color={probe_result.get('looks_color')}, "
            + f"mean_bgr={probe_result.get('best_mean_bgr')}"
        )

has_sdk_rgb = any("RGB Camera" in summary["sensors"] for summary in device_summaries)
has_present_v4l = any(node_info["present"] for node_info in realsense_v4l_nodes)
has_readable_v4l = any(probe_result.get("reads") for probe_result in v4l_probe_results)

if has_sdk_rgb:
    print("Status: RealSense SDK RGB is visible. You can use the preview and intrinsics cells.")
elif device_summaries and has_readable_v4l:
    print("Status: RealSense SDK sees the camera, but not its RGB sensor. The container can still read a RealSense V4L node.")
    print("That matches a 'guvcview works but pyrealsense2 is incomplete' situation.")
    print("Use that only for preview. Factory intrinsics and calibration still need 'RGB Camera' in the SDK sensor list.")
elif not device_summaries and has_present_v4l and has_readable_v4l:
    print("Status: pyrealsense2 sees no RealSense devices, but the container can still read a RealSense V4L node.")
    print("That usually means plain UVC access exists but the RealSense SDK is not attaching to the camera in this container.")
    print("This can happen after reboot even while guvcview on the host still works.")
elif not device_summaries and realsense_v4l_nodes and not has_present_v4l:
    print("Status: the kernel still exposes RealSense interfaces in /sys/class/video4linux, but the matching /dev/video* nodes are missing inside this container.")
    print("That points to a container device-node problem after reboot, not a notebook bug.")
    print("Reopen or rebuild the devcontainer after the camera is plugged in, or re-plug the camera so /dev/video* is recreated inside the container.")
elif not device_summaries and not realsense_v4l_nodes:
    print("Status: neither pyrealsense2 nor V4L can see a RealSense camera from this container.")
    print("That points to host USB enumeration, cable or power, or container passthrough.")
else:
    print("Status: RealSense state is inconsistent. Keep preview, intrinsics, and calibration blocked until this cell reports either SDK RGB or a readable V4L path.")

[ WARN:0@0.388] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name
[ WARN:0@0.388] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name
[ WARN:0@0.901] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name


RealSense SDK devices visible: 1
  Device 1: Intel RealSense D435 | serial=022422070965 | sensors=['Stereo Module', 'RGB Camera']
RealSense V4L nodes: [('video2', True, 'Intel(R) RealSense(TM) Depth Ca'), ('video3', True, 'Intel(R) RealSense(TM) Depth Ca'), ('video4', True, 'Intel(R) RealSense(TM) Depth Ca'), ('video5', True, 'Intel(R) RealSense(TM) Depth Ca'), ('video6', True, 'Intel(R) RealSense(TM) Depth Ca'), ('video7', True, 'Intel(R) RealSense(TM) Depth Ca')]
Container V4L probe:
  /dev/video2: opens=False, reads=False, fourcc=None, best_decode=None, channel_spread=None, looks_color=None, mean_bgr=None
  /dev/video3: opens=False, reads=False, fourcc=None, best_decode=None, channel_spread=None, looks_color=None, mean_bgr=None
  /dev/video4: opens=True, reads=True, fourcc=UYVY, best_decode=YUY2, channel_spread=315.5, looks_color=True, mean_bgr=[184.6, 75.2, 184.2]
  /dev/video5: opens=False, reads=False, fourcc=None, best_decode=None, channel_spread=None, looks_color=None, mean_bgr

[ WARN:0@1.788] global cap.cpp:215 open VIDEOIO(V4L2): backend is generally available but can't be used to capture by name


In [2]:
import time

import cv2
import numpy as np
import pyrealsense2 as rs

WIDTH = 1280
HEIGHT = 720
FPS = 30
PREVIEW_SECONDS = None  # Set to a number to auto-stop, or leave as None and press q in the preview window.

ctx = rs.context()
devices = list(ctx.query_devices())
if not devices:
    raise RuntimeError(
        "No RealSense device is visible to librealsense in this Docker container. "
        "Check USB passthrough and rerun the diagnostics cell first."
    )

device = devices[0]
device_name = device.get_info(rs.camera_info.name) if device.supports(rs.camera_info.name) else "RealSense camera"
serial_number = device.get_info(rs.camera_info.serial_number) if device.supports(rs.camera_info.serial_number) else None
sensor_names = [
    sensor.get_info(rs.camera_info.name)
    for sensor in device.query_sensors()
    if sensor.supports(rs.camera_info.name)
 ]
if "RGB Camera" not in sensor_names:
    raise RuntimeError(
        f"{device_name} is not exposing its RGB Camera sensor in this container. "
        f"Visible librealsense sensors: {sensor_names}. "
        "Host-side V4L apps like guvcview can still show a camera image, but this notebook cell uses the RealSense SDK RGB path. "
        "Use Cell 2 to inspect the container V4L nodes, but keep preview, intrinsics, and calibration on the SDK path until 'RGB Camera' appears here."
    )

pipeline = rs.pipeline()
config = rs.config()
if serial_number:
    config.enable_device(serial_number)
config.enable_stream(rs.stream.color, WIDTH, HEIGHT, rs.format.bgr8, FPS)

window_name = f"RealSense RGB preview - {device_name}"
print(f"Starting RGB preview from {device_name}...")
print("Press q in the preview window to stop, or set PREVIEW_SECONDS to auto-stop.")
pipeline_started = False
start_time = time.time()

try:
    pipeline.start(config)
    pipeline_started = True

    for _ in range(10):
        pipeline.wait_for_frames()

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    while True:
        frames = pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        if not color_frame:
            continue

        image = np.asanyarray(color_frame.get_data())
        cv2.imshow(window_name, image)

        key = cv2.waitKey(1) & 0xFF
        if key in (ord('q'), 27):
            print("Preview stopped by user.")
            break

        if PREVIEW_SECONDS is not None and (time.time() - start_time) >= PREVIEW_SECONDS:
            print("Preview finished.")
            break

finally:
    if pipeline_started:
        pipeline.stop()

Starting RGB preview from Intel RealSense D435...
Press q in the preview window to stop, or set PREVIEW_SECONDS to auto-stop.


QFontDatabase: Cannot find font directory /workspaces/welding_cell_ws/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /workspaces/welding_cell_ws/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /workspaces/welding_cell_ws/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /workspaces/welding_cell_ws/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find f

Preview stopped by user.


In [3]:
import pyrealsense2 as rs
import numpy as np

WIDTH = 1280
HEIGHT = 720
FPS = 30

ctx = rs.context()
devices = list(ctx.query_devices())
if not devices:
    raise RuntimeError(
        "No RealSense device is visible to librealsense in this Docker container. "
        "If the camera is physically connected, recreate or reconfigure the container with USB passthrough "
        "(for example /dev/bus/usb and udev metadata), then rerun this cell."
    )

device = devices[0]
device_name = device.get_info(rs.camera_info.name) if device.supports(rs.camera_info.name) else "RealSense camera"
serial_number = device.get_info(rs.camera_info.serial_number) if device.supports(rs.camera_info.serial_number) else None
sensor_names = [
    sensor.get_info(rs.camera_info.name)
    for sensor in device.query_sensors()
    if sensor.supports(rs.camera_info.name)
]
if "RGB Camera" not in sensor_names:
    raise RuntimeError(
        f"{device_name} is currently enumerated without its RGB Camera sensor. "
        f"Visible sensors: {sensor_names}. "
        "Color intrinsics and the ChArUco calibration cell require the RGB Camera to appear. "
        "Reconnect the camera on a USB 3 port/cable and verify the RGB sensor is visible before rerunning this cell."
    )

# 1. Initialize the RealSense pipeline
pipeline = rs.pipeline()
config = rs.config()
if serial_number:
    config.enable_device(serial_number)

# 2. Configure the RGB stream to match your SAM3 dataset resolution
config.enable_stream(rs.stream.color, WIDTH, HEIGHT, rs.format.bgr8, FPS)

print(f"Starting RealSense pipeline for {device_name}...")
pipeline_started = False

try:
    profile = pipeline.start(config)
    pipeline_started = True

    # Let the stream warm up before reading intrinsics.
    for _ in range(10):
        pipeline.wait_for_frames()

    # 3. Get the color stream profile
    color_stream = profile.get_stream(rs.stream.color)
    color_profile = rs.video_stream_profile(color_stream)

    # 4. Extract the intrinsic parameters from the firmware
    intrinsics = color_profile.get_intrinsics()

    # 5. Format it into the standard OpenCV 3x3 K-Matrix
    K_matrix = np.array([
        [intrinsics.fx, 0,             intrinsics.ppx],
        [0,             intrinsics.fy, intrinsics.ppy],
        [0,             0,             1]
    ], dtype=float)

    distortion_coeffs = np.array(intrinsics.coeffs[:5], dtype=float)

    print("\n=== RealSense Factory Intrinsics ===")
    print(f"Device: {device_name}")
    print(f"Visible sensors: {sensor_names}")
    print(f"Resolution: {intrinsics.width}x{intrinsics.height}")
    print(f"Focal Length (fx, fy): {intrinsics.fx:.2f}, {intrinsics.fy:.2f}")
    print(f"Principal Point (cx, cy): {intrinsics.ppx:.2f}, {intrinsics.ppy:.2f}")
    print(f"Distortion Model: {intrinsics.model}")
    print(f"Distortion Coefficients: {distortion_coeffs}")
    print("\nIntrinsic Matrix (K):")
    print(K_matrix)

    # Save this matrix to use in your Hand-Eye calibration script
    np.save("realsense_intrinsics.npy", K_matrix)
    print("\nSaved to 'realsense_intrinsics.npy'")

except RuntimeError as exc:
    raise RuntimeError(
        "Failed to start the RealSense RGB stream. Ensure the camera enumerates with an RGB Camera sensor, not only Stereo Module."
    ) from exc
finally:
    if pipeline_started:
        pipeline.stop()

Starting RealSense pipeline for Intel RealSense D435...

=== RealSense Factory Intrinsics ===
Device: Intel RealSense D435
Visible sensors: ['Stereo Module', 'RGB Camera']
Resolution: 1280x720
Focal Length (fx, fy): 919.47, 918.94
Principal Point (cx, cy): 650.85, 350.62
Distortion Model: distortion.inverse_brown_conrady
Distortion Coefficients: [0. 0. 0. 0. 0.]

Intrinsic Matrix (K):
[[919.46923828   0.         650.85241699]
 [  0.         918.93811035 350.61999512]
 [  0.           0.           1.        ]]

Saved to 'realsense_intrinsics.npy'


In [ ]:
# === Eye-in-Hand Hand-Eye Calibration (camera mounted on the UR5e wrist) ===
#
# Setup: the ChArUco board is FIXED on the table; the ARM moves the camera to
# many views. cv2.calibrateHandEye(gripper2base, target2cam) then returns
# cam2gripper, which is the CONSTANT transform  T_tool0_camera  (camera pose
# expressed in the tool0/flange frame). This is the eye-in-hand quantity we
# want; base->camera is NOT constant and is computed at runtime via TF as
#   T_base_camera(q) = T_base_tool0(q) @ T_tool0_camera.
#
# IMPORTANT: set the UR TCP to 0 (flange) before capturing, so
# getActualTCPPose() reports base->tool0. The pen-tip offset stays a separate
# transform (tool0->pen_tip) and must NOT be baked into this calibration.

import cv2
import numpy as np
import pyrealsense2 as rs
from rtde_receive import RTDEReceiveInterface

# --- Settings ---
ROBOT_IP = "192.168.8.4"
INTRINSICS_PATH = "realsense_intrinsics.npy"
SAMPLES_PATH = "handeye_samples.npz"          # raw samples (re-solvable offline)
EXTRINSIC_PATH = "T_tool0_camera.npy"          # <-- eye-in-hand result

# ChArUco board physical size (metres) — fixed on the table
SQUARES_X = 4
SQUARES_Y = 4
SQUARE_LENGTH = 0.030   # black square edge
MARKER_LENGTH = 0.022   # inner ArUco marker edge

# --- OpenCV ArUco / ChArUco (OpenCV 4.7+) ---
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_6X6_250)
board = cv2.aruco.CharucoBoard((SQUARES_X, SQUARES_Y), SQUARE_LENGTH, MARKER_LENGTH, aruco_dict)
aruco_detector = cv2.aruco.ArucoDetector(aruco_dict, cv2.aruco.DetectorParameters())

K_matrix = np.load(INTRINSICS_PATH)
dist_coeffs = np.zeros((5, 1))  # RealSense factory distortion is ~0

# --- Connections ---
print("Connecting to robot...")
rtde_r = RTDEReceiveInterface(ROBOT_IP)

print("Starting camera...")
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 1280, 720, rs.format.bgr8, 30)
pipeline.start(config)

# Sample buffers
R_gripper2base, t_gripper2base = [], []
R_board2cam, t_board2cam = [], []

print("\n--- EYE-IN-HAND CAPTURE ---")
print("Board FIXED on table; move the ARM. Vary pitch/yaw/roll widely.")
print("'c' = capture (board must be clearly seen), 'q' = finish & solve. Aim for 15+ diverse poses.")

try:
    while True:
        frames = pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        if not color_frame:
            continue

        image = np.asanyarray(color_frame.get_data())
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        disp = image.copy()

        corners, ids, _ = aruco_detector.detectMarkers(gray)
        board_found = False
        rvec = tvec = None

        if ids is not None and len(ids) > 0:
            cv2.aruco.drawDetectedMarkers(disp, corners, ids)
            ret, ch_corners, ch_ids = cv2.aruco.interpolateCornersCharuco(
                corners, ids, gray, board)
            if ch_corners is not None and ch_ids is not None and len(ch_corners) > 3:
                cv2.aruco.drawDetectedCornersCharuco(disp, ch_corners, ch_ids, (0, 255, 0))
                ok, rvec, tvec = cv2.aruco.estimatePoseCharucoBoard(
                    ch_corners, ch_ids, board, K_matrix, dist_coeffs, None, None)
                if ok:
                    cv2.drawFrameAxes(disp, K_matrix, dist_coeffs, rvec, tvec, 0.05)
                    board_found = True

        cv2.putText(disp, f"samples: {len(R_gripper2base)} (need 15+)", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
        cv2.imshow("Eye-in-Hand Calibration (c: capture, q: solve)", disp)
        key = cv2.waitKey(1) & 0xFF

        if key == ord('c') and board_found:
            # Robot pose: base -> tool0 (UR base frame). TCP MUST be 0 here.
            tcp = rtde_r.getActualTCPPose()
            Rg, _ = cv2.Rodrigues(np.array(tcp[3:]))
            R_gripper2base.append(Rg)
            t_gripper2base.append(np.array(tcp[:3]).reshape(3, 1))
            # Board pose: board -> camera
            Rc, _ = cv2.Rodrigues(rvec)
            R_board2cam.append(Rc)
            t_board2cam.append(np.array(tvec).reshape(3, 1))
            print(f"[{len(R_gripper2base)}] captured")

        elif key == ord('q'):
            break
finally:
    pipeline.stop()
    cv2.destroyAllWindows()

n = len(R_gripper2base)
if n < 5:
    raise RuntimeError(f"Need >=5 samples to solve (ideally 15+); got {n}.")

# Save raw samples so the calibration can be re-solved / verified offline.
np.savez(
    SAMPLES_PATH,
    R_gripper2base=np.array(R_gripper2base),
    t_gripper2base=np.array(t_gripper2base),
    R_board2cam=np.array(R_board2cam),
    t_board2cam=np.array(t_board2cam),
)
print(f"\nSaved {n} raw samples to '{SAMPLES_PATH}'.")

# --- Solve AX = XB (eye-in-hand) -> cam2gripper == T_tool0_camera ---
R_cam2gripper, t_cam2gripper = cv2.calibrateHandEye(
    R_gripper2base, t_gripper2base,
    R_board2cam, t_board2cam,
    method=cv2.CALIB_HAND_EYE_TSAI,
)

T_tool0_camera = np.eye(4)
T_tool0_camera[:3, :3] = R_cam2gripper
T_tool0_camera[:3, 3] = t_cam2gripper.flatten()

np.save(EXTRINSIC_PATH, T_tool0_camera)
print("\n=== RESULT: T_tool0_camera (camera pose in the tool0/flange frame) ===")
print(np.round(T_tool0_camera, 4))
print(f"Camera origin offset from flange: {np.linalg.norm(t_cam2gripper) * 1000:.1f} mm")
print(f"Saved to '{EXTRINSIC_PATH}'.")
print("\nNext: publish this as a static TF  tool0 -> camera_color_optical_frame,")
print("then base->object = TF(base_link->tool0) @ T_tool0_camera @ pose_camera.")
print("Run the next cell to check calibration quality before trusting it.")


In [ ]:
# === Verify calibration quality (run after capture, or standalone) ===
#
# Eye-in-hand invariant: the board is fixed on the table, so for every sample
#   T_base_board = T_base_tool0 @ T_tool0_camera @ T_camera_board
# must be (nearly) the SAME rigid pose. The spread of that reconstructed board
# pose across samples is a direct, unit-ful quality metric:
#   - position spread  (mm)  : how much the board "moves" due to calibration error
#   - rotation spread  (deg) : orientation consistency
# Rules of thumb (D435 @ ~0.5 m): < ~5 mm and < ~1 deg is good; larger means
# add more diverse orientations or re-capture. All four solvers are compared so
# you can pick the most consistent one; TSAI is what the capture cell saved.

import cv2
import numpy as np

data = np.load("handeye_samples.npz")
Rg2b = list(data["R_gripper2base"])
tg2b = list(data["t_gripper2base"])
Rb2c = list(data["R_board2cam"])
tb2c = list(data["t_board2cam"])
n = len(Rg2b)
print(f"Loaded {n} samples.\n")


def homog(R, t):
    T = np.eye(4)
    T[:3, :3] = np.asarray(R)
    T[:3, 3] = np.asarray(t).flatten()
    return T


def rot_angle_deg(R):
    return np.degrees(np.arccos(np.clip((np.trace(R) - 1.0) / 2.0, -1.0, 1.0)))


methods = [
    ("TSAI", cv2.CALIB_HAND_EYE_TSAI),
    ("PARK", cv2.CALIB_HAND_EYE_PARK),
    ("HORAUD", cv2.CALIB_HAND_EYE_HORAUD),
    ("DANIILIDIS", cv2.CALIB_HAND_EYE_DANIILIDIS),
]

print(f"{'method':<12}{'cam_offset(mm)':>16}{'pos_spread(mm)':>16}{'rot_spread(deg)':>17}")
best = None
for name, method in methods:
    Rx, tx = cv2.calibrateHandEye(Rg2b, tg2b, Rb2c, tb2c, method=method)
    T_tc = homog(Rx, tx)
    boards = [homog(Rg2b[i], tg2b[i]) @ T_tc @ homog(Rb2c[i], tb2c[i]) for i in range(n)]
    pos = np.array([b[:3, 3] for b in boards])
    pos_spread_mm = float(np.linalg.norm(pos.std(axis=0)) * 1000.0)
    ref_R = boards[0][:3, :3]
    ang = [rot_angle_deg(ref_R.T @ b[:3, :3]) for b in boards]
    rot_spread_deg = float(np.std(ang))
    cam_offset_mm = float(np.linalg.norm(tx) * 1000.0)
    print(f"{name:<12}{cam_offset_mm:>16.1f}{pos_spread_mm:>16.2f}{rot_spread_deg:>17.3f}")
    score = pos_spread_mm + 10.0 * rot_spread_deg
    if best is None or score < best[0]:
        best = (score, name, T_tc, pos_spread_mm, rot_spread_deg)

_, best_name, best_T, best_pos, best_rot = best
print(f"\nMost consistent solver: {best_name}  "
      f"(pos spread {best_pos:.2f} mm, rot spread {best_rot:.3f} deg)")
print("T_tool0_camera (best):")
print(np.round(best_T, 4))
if best_pos > 5.0 or best_rot > 1.0:
    print("\nWARNING: spread is high -> collect more diverse orientations and re-capture.")
# Uncomment to overwrite the saved extrinsic with the most consistent solver:
# np.save("T_tool0_camera.npy", best_T)
